In [4]:
pip install openmeteo-requests
pip install requests-cache retry-requests numpy pandas

SyntaxError: invalid syntax (1955238504.py, line 1)

In [9]:

import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# 1. API client setup
cache_session = requests_cache.CachedSession(
    ".cache",
    expire_after=-1
)

retry_session = retry(
    cache_session,
    retries=5,
    backoff_factor=0.2
)

openmeteo = openmeteo_requests.Client(
    session=retry_session
)

# 2. API URL
url = "https://archive-api.open-meteo.com/v1/archive"

# 3. Parameters
params = {
    "latitude": [
        40.4093, 39.2089, 40.55, 40.65, 40.55, 39.9, 41.05, 41.35, 38.75, 40.55, 39.9, 41.25, 39.2, 39.9
    ],
    "longitude": [
        49.8671, 45.4122, 49.55, 48.6, 46.5, 47.1, 45.4, 48.5, 48.85, 47.4, 48.2, 47.2, 46.6, 48.9
    ],
    "start_date": "2016-01-01",
    "end_date": "2026-11-03",
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean"
    ],
    "timezone": "auto",
}

# 4. Get weather data
responses = openmeteo.weather_api(url, params=params)

# 5. Store all locations in one list
all_data = []

for location_id, response in enumerate(responses, start=1):

    print(f"Processing location {location_id}...")

    daily = response.Daily()

    # Extract daily weather variables
    daily_max = daily.Variables(0).ValuesAsNumpy()
    daily_min = daily.Variables(1).ValuesAsNumpy()
    daily_mean = daily.Variables(2).ValuesAsNumpy()

    # Generate dates
    dates = pd.date_range(
        start=pd.to_datetime(daily.Time(), unit="s", utc=True),
        end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive="left"
    ).tz_convert(response.Timezone().decode())

    # Create a DataFrame for this location
    daily_dataframe = pd.DataFrame({
        "location_id": location_id,
        "latitude": response.Latitude(),
        "longitude": response.Longitude(),
        "date": dates.strftime("%Y-%m-%d"),
        "temperature_2m_max": daily_max,
        "temperature_2m_min": daily_min,
        "temperature_2m_mean": daily_mean
    })

    all_data.append(daily_dataframe)

# 6. Combine all locations into one DataFrame
weather = pd.concat(all_data, ignore_index=True)

print("Total rows:", len(weather_df))


Processing location 1...
Processing location 2...
Processing location 3...
Processing location 4...
Processing location 5...
Processing location 6...
Processing location 7...
Processing location 8...
Processing location 9...
Processing location 10...
Processing location 11...
Processing location 12...
Processing location 13...
Processing location 14...

CSV file created successfully!
File name: weather_azerbaijan.csv
Total rows: 55440
   location_id  latitude  longitude        date  temperature_2m_max  \
0            1  40.45694  49.887169  2016-01-01            4.974000   
1            1  40.45694  49.887169  2016-01-02            3.974000   
2            1  40.45694  49.887169  2016-01-03            2.724000   
3            1  40.45694  49.887169  2016-01-04            4.162917   
4            1  40.45694  49.887169  2016-01-05            7.233601   

   temperature_2m_min  temperature_2m_mean  
0            2.624000             3.957333  
1            0.574000             2.321917  

In [11]:
weather.head()

,location_id,latitude,longitude,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean
0,1,40.45694,49.887169,2016-01-01,4.974000,2.624000,3.957333
1,1,40.45694,49.887169,2016-01-02,3.974000,0.574000,2.321917
2,1,40.45694,49.887169,2016-01-03,2.724000,-0.337647,1.207333
3,1,40.45694,49.887169,2016-01-04,4.162917,0.024000,1.342750
4,1,40.45694,49.887169,2016-01-05,7.233601,2.730758,4.746917


In [12]:
print(weather.shape)
print(weather.describe)

(55440, 7)
<bound method DataFrame.describe of        location_id  latitude  longitude        date  temperature_2m_max  \
0                1  40.45694  49.887169  2016-01-01            4.974000   
1                1  40.45694  49.887169  2016-01-02            3.974000   
2                1  40.45694  49.887169  2016-01-03            2.724000   
3                1  40.45694  49.887169  2016-01-04            4.162917   
4                1  40.45694  49.887169  2016-01-05            7.233601   
...            ...       ...        ...         ...                 ...   
55435           14  39.89455  48.953976  2026-10-30           17.786861   
55436           14  39.89455  48.953976  2026-10-31           20.666819   
55437           14  39.89455  48.953976  2026-11-01           20.924999   
55438           14  39.89455  48.953976  2026-11-02           20.138390   
55439           14  39.89455  48.953976  2026-11-03           20.201647   

       temperature_2m_min  temperature_2m_mean  
0  

In [21]:
#Rename the columns
weather = weather.rename(columns={
    "temperature_2m_mean": "temperature_mean_c",
    "temperature_2m_max": "temperature_max_c",
    "temperature_2m_min": "temperature_min_c" })

In [14]:
#Convert the date column
weather["date"] = pd.to_datetime(
    weather["date"],
    errors="coerce"  )

In [15]:
print(weather["date"].dtype)
print("Invalid dates:", weather["date"].isna().sum())

print(weather.duplicated().sum()) # Check duplicate records

datetime64[us]
Invalid dates: 0
0


In [16]:
weather = weather.drop_duplicates()

In [17]:
#Check the date range
print("Start date:", weather["date"].min())
print("End date:", weather["date"].max())

Start date: 2016-01-01 00:00:00
End date: 2026-11-03 00:00:00


In [18]:
#Check for missing dates
for location_id, group in weather.groupby("location_id"):
    dates = pd.DatetimeIndex(
        group["date"].dropna().unique()
    ).sort_values()

    expected_dates = pd.date_range(
        start=dates.min(),
        end=dates.max(),
        freq="D"
    )

    missing_dates = expected_dates.difference(dates)

    print(
        f"Location {location_id}: "
        f"{len(missing_dates)} missing dates"
    )

Location 1: 0 missing dates
Location 2: 0 missing dates
Location 3: 0 missing dates
Location 4: 0 missing dates
Location 5: 0 missing dates
Location 6: 0 missing dates
Location 7: 0 missing dates
Location 8: 0 missing dates
Location 9: 0 missing dates
Location 10: 0 missing dates
Location 11: 0 missing dates
Location 12: 0 missing dates
Location 13: 0 missing dates
Location 14: 0 missing dates


In [22]:
#For the same location and day weather should be: Tmin < Tmean < Tmax.

#Check temperature consistency:

invalid_temperature = weather[
    (weather["temperature_min_c"] >
     weather["temperature_mean_c"])
    |
    (weather["temperature_mean_c"] >
     weather["temperature_max_c"])
    ]

print(invalid_temperature)

Empty DataFrame
Columns: [location_id, latitude, longitude, date, temperature_max_c, temperature_min_c, temperature_mean_c]
Index: []


In [23]:
# Investigate extreme temperatures
suspicious_temperatures = weather[
    (weather["temperature_min_c"] < -40)   |    (weather["temperature_max_c"] > 50)
    ]

print(suspicious_temperatures)

Empty DataFrame
Columns: [location_id, latitude, longitude, date, temperature_max_c, temperature_min_c, temperature_mean_c]
Index: []


In [25]:
#Save
weather.to_csv("weather.csv", index=False, date_format="%Y-%m-%d")